# PY500 — Sampling and Population Estimation

We rarely have access to an entire **population** (all possible data points). Instead, we work with **samples** and use them to make inferences about the population.

## Topics Covered
1. Sampling methods (Random, Stratified, Systematic, Cluster)
2. Sample size determination
3. Point estimates and Confidence Intervals
4. Standard Error
5. Bootstrap resampling

> **Perspective:** The entire field of inferential statistics rests on one idea — a well-chosen sample can tell you about the population. The quality of your sample matters more than its size. A biased sample of 1 million is worse than an unbiased sample of 1,000.

---
## 1. Sampling Methods

| Method | How It Works | Best For |
|---|---|---|
| **Simple Random** | Every item has equal probability | General purpose |
| **Stratified** | Divide into groups, sample from each | Ensuring minority representation |
| **Systematic** | Every k-th item | Production lines, time series |
| **Cluster** | Randomly select entire groups | Geographic surveys |

In [ ]:
## Sampling demonstrations

import numpy as np
import pandas as pd

np.random.seed(42)

# Create a population
population = pd.DataFrame({
    'id': range(1, 1001),
    'dept': np.random.choice(['IT', 'HR', 'Finance', 'Ops'], 1000, p=[0.4, 0.2, 0.15, 0.25]),
    'salary': np.random.normal(60000, 15000, 1000).clip(25000)
})

print(f"Population size: {len(population)}")
print(f"Population mean salary: ₹{population['salary'].mean():,.0f}")
print(f"Dept distribution:\n{population['dept'].value_counts(normalize=True).round(3)}")

In [ ]:
## Simple Random Sampling
simple_sample = population.sample(n=50, random_state=42)
print(f"Simple Random (n=50) mean: ₹{simple_sample['salary'].mean():,.0f}")

## Stratified Sampling — proportional representation of each dept
stratified = population.groupby('dept', group_keys=False).apply(
    lambda x: x.sample(frac=0.05, random_state=42)    # 5% from each group
)
print(f"\nStratified (5% each) mean: ₹{stratified['salary'].mean():,.0f}")
print(f"Stratified dept distribution:\n{stratified['dept'].value_counts(normalize=True).round(3)}")

## Systematic Sampling — every k-th item
k = len(population) // 50   # Select ~50 items
systematic = population.iloc[::k]
print(f"\nSystematic (every {k}th) mean: ₹{systematic['salary'].mean():,.0f} (n={len(systematic)})")

---
## 2. Confidence Intervals

A **confidence interval** gives a range that likely contains the true population parameter.

```
CI = sample_mean ± z * (std / √n)
```

A 95% CI means: if we repeated this sampling 100 times, about 95 of those intervals would contain the true mean.

In [ ]:
## Confidence Interval calculation

from scipy import stats

sample = population.sample(n=50, random_state=42)
sample_mean = sample['salary'].mean()
sample_std = sample['salary'].std()
n = len(sample)

# Standard Error = std / sqrt(n)
se = sample_std / np.sqrt(n)

# 95% Confidence Interval using t-distribution (better for small samples)
ci_95 = stats.t.interval(0.95, df=n-1, loc=sample_mean, scale=se)
ci_99 = stats.t.interval(0.99, df=n-1, loc=sample_mean, scale=se)

pop_mean = population['salary'].mean()

print(f"Population mean  : ₹{pop_mean:,.0f}")
print(f"Sample mean      : ₹{sample_mean:,.0f}")
print(f"Standard Error   : ₹{se:,.0f}")
print(f"95% CI           : [₹{ci_95[0]:,.0f}, ₹{ci_95[1]:,.0f}]")
print(f"99% CI           : [₹{ci_99[0]:,.0f}, ₹{ci_99[1]:,.0f}]")
print(f"Pop mean in 95%? : {ci_95[0] <= pop_mean <= ci_95[1]}")

# Learning Point: Higher confidence = wider interval.
# 99% CI is wider than 95% CI because you need a bigger net to be more sure.

---
## 3. Bootstrap Resampling

When you don't know the population distribution, **bootstrap** estimates it by resampling *with replacement* from your sample. No assumptions about normality needed.

In [ ]:
## Bootstrap confidence interval

import matplotlib.pyplot as plt

sample = population.sample(n=50, random_state=42)
n_bootstrap = 10000
boot_means = []

for _ in range(n_bootstrap):
    boot_sample = sample['salary'].sample(n=len(sample), replace=True)
    boot_means.append(boot_sample.mean())

boot_means = np.array(boot_means)

# Bootstrap 95% CI = 2.5th and 97.5th percentiles
ci_lower = np.percentile(boot_means, 2.5)
ci_upper = np.percentile(boot_means, 97.5)

print(f"Bootstrap 95% CI: [₹{ci_lower:,.0f}, ₹{ci_upper:,.0f}]")
print(f"Population mean : ₹{population['salary'].mean():,.0f}")

plt.figure(figsize=(10, 4))
plt.hist(boot_means, bins=50, color='steelblue', edgecolor='white', density=True)
plt.axvline(ci_lower, color='red', linestyle='--', label=f'2.5%: ₹{ci_lower:,.0f}')
plt.axvline(ci_upper, color='red', linestyle='--', label=f'97.5%: ₹{ci_upper:,.0f}')
plt.axvline(population['salary'].mean(), color='green', linewidth=2, label='True mean')
plt.title('Bootstrap Distribution of Sample Means')
plt.legend()
plt.tight_layout()
plt.show()

# Best Practice: Bootstrap is non-parametric — no normality assumption.
# It works for means, medians, ratios, correlations — any statistic.

> **Key Takeaway:** The Standard Error shrinks with √n. To halve the SE, you need 4x the sample size. This is the law of diminishing returns in sampling — going from n=100 to n=10,000 improves precision by only 10x, not 100x.